# Huấn Luyện Matcha-TTS Tiếng Việt: Giọng Mượt & Chuẩn Thanh Điệu

Notebook này khắc phục các nguyên nhân khiến giọng của bản train trước (VIVOS) chưa tự nhiên:

| Vấn đề ở bản VIVOS | Cách xử lý |
|---|---|
| VIVOS là dữ liệu ASR: 16 kHz, nhiễu nền (~ -37 dB), 65 giọng × ~14 phút | **InfoRe1**: 25 giờ, **một giọng**, thu sạch (~ -52 dB). Tùy chọn PhoAudiobook (941 giờ) |
| Token = ký tự mang dấu (91 token), thanh điệu bị trộn với nguyên âm | **Tách thanh điệu** thành token cuối âm tiết (`hồ` → `h ô 2`), 5 thanh dùng chung cho mọi nguyên âm |
| Chuẩn hóa âm lượng theo đỉnh | Chuẩn hóa theo năng lượng (RMS) → âm lượng các câu đồng đều |
| Dừng ở ~15k step do early stopping theo val loss | Train theo **số step** (mặc định 150k), lưu snapshot định kỳ, **chọn bằng CER** (ASR PhoWhisper) + nghe |
| HiFi-GAN Universal chưa quen mel do Matcha sinh ra → rè, kim loại | **Fine-tune HiFi-GAN** trên mel teacher-forced của chính mô hình |

**Cách dùng:** `Runtime → Change runtime type → T4 GPU` (L4/A100 nhanh hơn) rồi `Runtime → Run all`. Mất kết nối thì **Run all** lại, mọi bước tự tiếp tục từ chỗ dừng.

**Lưu trữ:** mọi file nằm trong `MyDrive/Colab Notebooks/TTS/` (thư mục `Colab Notebooks` do Colab tự tạo). Notebook tự tạo `TTS` và các thư mục con:

```
MyDrive/Colab Notebooks/TTS/
├── data/        dữ liệu đã tiền xử lý: mel, wav cho vocoder, meta (tạo một lần)
├── training/    trạng thái resume, tự ghi đè trong lúc train
├── snapshots/   step_XXXXXX.pt: snapshot Matcha định kỳ
├── samples/     file wav nghe thử
└── export/      KẾT QUẢ CUỐI: matcha_vi_v2.pt, matcha_vi_v2_hifigan.pt
```

**Thời gian ước tính trên T4** (chưa đo thực tế, xem số phút/epoch ở mục 6 để tính lại): chuẩn bị dữ liệu 30–45 phút (một lần) · train 100k step ~8–10 giờ · chọn checkpoint ~10 phút · fine-tune vocoder 30k step ~3 giờ. Colab miễn phí cấp GPU theo phiên ngắn và hạn mức không cố định, nên thường phải chia ra **nhiều ngày**. Có Colab Pro thì có thể tăng `TOTAL_STEPS` lên 150k.

**Dung lượng Drive cần khoảng 6 GB.** File bị ghi đè hoặc xóa trên Drive có thể nằm lại trong **Thùng rác** và vẫn tính vào dung lượng. Nếu Drive báo đầy, hãy dọn Thùng rác (drive.google.com → Thùng rác → Dọn sạch).

**Kết quả cần tải về** `checkpoints/` của dự án: `matcha_vi_v2.pt` và `matcha_vi_v2_hifigan.pt`. `src/matcha_engine.py` tự ưu tiên hai file này.

## 1. Cấu Hình

In [ ]:
import os
import sys

# Chạy thử nhanh trên vài chục câu để kiểm tra code (không dùng để train thật)
SMOKE_TEST = bool(os.environ.get('MATCHA_SMOKE_TEST'))

DATASET = 'infore1'
DATASETS = {
    # InfoRe Technology 1: ~25 giờ, 1 giọng nữ, thu sạch, công khai -> lựa chọn tốt nhất cho Colab
    'infore1': dict(repo='doof-ferb/infore1_25hours', text_col='transcription', audio_col='audio',
                    speaker_col=None, shard_prefix='data/train-'),
    # PhoAudiobook (ACL 2025): 941 giờ sách nói, 735 giọng, chỉ dùng cho nghiên cứu/giáo dục.
    # Cần bấm đồng ý điều khoản trên trang dataset và thêm HF_TOKEN vào Colab Secrets (biểu tượng chìa khóa).
    'phoaudiobook': dict(repo='thivux/phoaudiobook', text_col='text', audio_col='audio',
                         speaker_col='speaker', shard_prefix='data/train-'),
}
MAX_SHARDS = None   # None = tất cả shard (infore1: 15 shard ~7.8 GB). PhoAudiobook có 803 shard -> đặt ví dụ 40
MAX_UTTS = None     # giới hạn số câu (None = không giới hạn)
SPEAKERS = None     # dataset nhiều giọng: None = tự chọn 1 giọng nhiều dữ liệu nhất, hoặc liệt kê tên giọng
MIN_SECONDS, MAX_SECONDS = 1.0, 12.0
N_VAL = 100
N_WORKERS = os.cpu_count()

# Huấn luyện Matcha (không early stopping: val loss của flow matching chững sớm dù giọng vẫn còn cải thiện)
TOTAL_STEPS = 100_000     # Colab Pro: có thể tăng lên 150_000
SNAPSHOT_EVERY = 10_000   # lưu snapshot để chọn bằng CER + nghe ở mục 7
SAMPLE_EVERY = 5_000      # phát thử câu mẫu
LEARNING_RATE = 1e-4
GRAD_CLIP = 5.0
N_EVAL_SNAPSHOTS = 6      # số snapshot mới nhất được chấm CER ở mục 7
# Lưu trạng thái resume theo thời gian (file lớn: Matcha ~220 MB, vocoder ~1 GB). Lưu quá dày sẽ làm đầy
# Thùng rác Drive vì mỗi lần ghi đè có thể để lại bản cũ; mất kết nối thì chỉ mất tối đa chừng này phút.
RESUME_EVERY_MIN = 20
VOC_RESUME_EVERY_MIN = 30

# Fine-tune vocoder HiFi-GAN
VOC_N_UTTS = 3000         # số câu (~5 giờ) dùng fine-tune vocoder
VOC_EVERY = 4             # lấy 1 câu trong mỗi 4 câu để rải đều trên dataset
VOC_STEPS = 30_000
VOC_BATCH = 16
VOC_LR = 1e-4
VOC_DISC_WARMUP = 2_000   # discriminator học từ đầu: vài nghìn step đầu generator chỉ dùng mel loss
VOC_SAMPLE_EVERY = 10_000
VOC_TF_TIMESTEPS = 10     # số bước ODE khi sinh mel teacher-forced

if SMOKE_TEST:
    MAX_SHARDS, MAX_UTTS, N_VAL, N_WORKERS = 1, 60, 6, 0
    TOTAL_STEPS, SNAPSHOT_EVERY, SAMPLE_EVERY, N_EVAL_SNAPSHOTS = 12, 6, 6, 2
    VOC_N_UTTS, VOC_EVERY, VOC_STEPS, VOC_BATCH = 6, 2, 6, 2
    VOC_DISC_WARMUP, VOC_SAMPLE_EVERY = 3, 6
    RESUME_EVERY_MIN = VOC_RESUME_EVERY_MIN = 0
print(f'[OK] Dataset: {DATASET} | {TOTAL_STEPS} step | smoke test: {SMOKE_TEST}')

## 2. GPU & Google Drive

In [ ]:
import subprocess

import torch

HAS_GPU = torch.cuda.is_available()
if not HAS_GPU and not SMOKE_TEST:
    raise RuntimeError('Chua bat GPU! Vao Runtime -> Change runtime type -> chon T4 GPU, roi chay lai.')
DEVICE = 'cuda' if HAS_GPU else 'cpu'
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3 if HAS_GPU else 0.0
if HAS_GPU:
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | VRAM {GPU_VRAM_GB:.1f} GB | compute capability {gpu.major}.{gpu.minor}')
BATCH_SIZE = 4 if SMOKE_TEST else (64 if GPU_VRAM_GB > 30 else 32)   # giảm xuống 16 nếu CUDA out of memory

try:
    from google.colab import drive

    drive.mount('/content/drive')
    # Đặt trong thư mục "Colab Notebooks" mà Colab tự tạo trên Drive; TTS là thư mục gốc của dự án
    DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks/TTS'
    WORK_DIR = '/content'
except ImportError:  # chạy ngoài Colab (smoke test cục bộ)
    DRIVE_DIR = os.environ.get('MATCHA_DRIVE_DIR', os.path.abspath('matcha_v2_drive'))
    WORK_DIR = os.environ.get('MATCHA_WORK_DIR', os.path.abspath('matcha_v2_work'))

# Cấu trúc thư mục trong MyDrive/Colab Notebooks/TTS (tự tạo nếu chưa có):
#   data/       dữ liệu đã tiền xử lý (tạo một lần, các phiên sau chỉ nạp lại)
#   training/   trạng thái resume (tự ghi đè trong lúc train)
#   snapshots/  snapshot Matcha định kỳ, dùng để chọn ở mục 7
#   samples/    file wav nghe thử
#   export/     KẾT QUẢ CUỐI: tải 2 file trong này về checkpoints/ của dự án
DATA_DIR, TRAIN_DIR, SNAPSHOT_DIR, SAMPLES_DIR, EXPORT_DIR = (
    f'{DRIVE_DIR}/{name}' for name in ('data', 'training', 'snapshots', 'samples', 'export'))

DATA_PACK = f'{DATA_DIR}/{DATASET}_mels_22k.pt'        # mel đã tiền xử lý
VOC_PACK = f'{DATA_DIR}/{DATASET}_voc_wavs.pt'         # wav cho fine-tune vocoder
META_PATH = f'{DATA_DIR}/{DATASET}_meta.json'          # bộ token, frontend, giọng, thống kê mel
VOC_TF_PACK = f'{DATA_DIR}/voc_teacher_forced.pt'      # mel teacher-forced cho vocoder
RESUME_CKPT = f'{TRAIN_DIR}/matcha_v2_resume.pt'       # trạng thái train Matcha mới nhất (model + optimizer)
VOC_RESUME = f'{TRAIN_DIR}/hifigan_v2_resume.pt'       # trạng thái fine-tune vocoder mới nhất
FINAL_CKPT = f'{EXPORT_DIR}/matcha_vi_v2.pt'           # snapshot được chọn ở mục 7 (step_XXXXXX.pt ~75 MB)
FINAL_VOC = f'{EXPORT_DIR}/matcha_vi_v2_hifigan.pt'    # vocoder fine-tune, đi kèm FINAL_CKPT
for d in (DATA_DIR, TRAIN_DIR, SNAPSHOT_DIR, SAMPLES_DIR, EXPORT_DIR, WORK_DIR):
    os.makedirs(d, exist_ok=True)
import shutil

free_gb = shutil.disk_usage(DRIVE_DIR).free / 1024**3
print(f'[OK] Luu tru: {DRIVE_DIR} ({free_gb:.1f} GB trong) | device {DEVICE} | batch {BATCH_SIZE}')
if free_gb < 8:
    print('[CANH BAO] Drive con it hon 8 GB: hay don Thung rac / xoa file cu truoc khi train.')

## 3. Cài Đặt Matcha-TTS

Clone mã nguồn (không `pip install` vì cần biên dịch Cython) và thay Monotonic Alignment Search bằng bản **numba**.

In [ ]:
MATCHA_DIR = os.environ.get('MATCHA_DIR', f'{WORK_DIR}/Matcha-TTS')
if not os.path.exists(MATCHA_DIR):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/shivammehta25/Matcha-TTS.git', MATCHA_DIR],
                   check=True)
if not SMOKE_TEST:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightning', 'hydra-core', 'conformer==0.3.2',
                    'diffusers', 'einops', 'wget', 'gdown', 'numba', 'huggingface_hub', 'pyarrow',
                    'transformers'], check=True)

# Monotonic Alignment Search viết bằng numba, thay cho C-extension Cython của bản gốc
MAS_CODE = """import numba
import numpy as np
import torch


@numba.njit(parallel=True, cache=True)
def _maximum_path_batch(paths, values, t_xs, t_ys):
    max_neg_val = -1e9
    for b in numba.prange(paths.shape[0]):
        path, value, t_x, t_y = paths[b], values[b], t_xs[b], t_ys[b]
        for y in range(t_y):
            for x in range(max(0, t_x + y - t_y), min(t_x, y + 1)):
                v_cur = max_neg_val if x == y else value[x, y - 1]
                if x == 0:
                    v_prev = 0.0 if y == 0 else max_neg_val
                else:
                    v_prev = value[x - 1, y - 1]
                value[x, y] += max(v_prev, v_cur)
        index = t_x - 1
        for y in range(t_y - 1, -1, -1):
            path[index, y] = 1
            if index != 0 and (index == y or value[index, y - 1] < value[index - 1, y - 1]):
                index -= 1


def maximum_path(value, mask):
    value = value * mask
    device, dtype = value.device, value.dtype
    value_np = value.detach().float().cpu().numpy()
    mask_np = mask.detach().cpu().numpy()
    path = np.zeros_like(value_np, dtype=np.int32)
    t_x = mask_np.sum(1)[:, 0].astype(np.int32)
    t_y = mask_np.sum(2)[:, 0].astype(np.int32)
    _maximum_path_batch(path, value_np, t_x, t_y)
    return torch.from_numpy(path).to(device=device, dtype=dtype)
"""
with open(f'{MATCHA_DIR}/matcha/utils/monotonic_align/__init__.py', 'w', encoding='utf-8') as f:
    f.write(MAS_CODE)

for p in (MATCHA_DIR, os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
from matcha.models.matcha_tts import MatchaTTS  # kiểm tra import

print('[OK] Matcha-TTS san sang.')

## 4. Frontend Tiếng Việt (tách thanh điệu)

Cell dưới ghi ra `vi_frontend.py`, **bản sao nguyên văn** của `src/vi_frontend.py` trong dự án, để train và suy luận token hóa giống hệt nhau. Không sửa trực tiếp cell này: sửa `src/vi_frontend.py` rồi chạy `python scripts/sync_notebook_frontend.py` (`tests/test_vi_frontend.py` kiểm tra hai bản khớp nhau).

In [ ]:
%%writefile vi_frontend.py
"""Frontend văn bản tiếng Việt cho Matcha-TTS, dùng chung cho notebook huấn luyện và engine suy luận.

Hai kiểu token hóa (lưu trong metadata checkpoint, khóa "frontend"):
- "char_v1": mỗi ký tự NFC là một token (nguyên âm mang dấu thanh là token riêng). Dùng cho checkpoint VIVOS cũ.
- "vi_tone_v2": tách dấu thanh khỏi nguyên âm và đặt thành token cuối mỗi âm tiết
  ("hồ" -> h, ô, 2). Năm thanh dùng chung cho mọi nguyên âm nên học thanh điệu nhanh và chuẩn hơn;
  dấu câu (, . ? !) được giữ làm token để mô hình học ngắt nghỉ, ngữ điệu (nếu dữ liệu có dấu câu).

Notebook notebooks/02_train_matcha_colab.ipynb chứa bản sao nguyên văn file này (kiểm tra bởi
tests/test_vi_frontend.py). Sau khi sửa file này, chạy `python scripts/sync_notebook_frontend.py`.
"""

import re
import unicodedata
from typing import Dict, List

# Tên chữ cái tiếng Việt, dùng để đánh vần nhãn đáp án (A/B/C/D) và chữ viết tắt
LETTER_NAMES = {
    "a": "a", "b": "bê", "c": "xê", "d": "dê", "đ": "đê", "e": "e", "f": "ép", "g": "giê",
    "h": "hát", "i": "i", "j": "gi", "k": "ca", "l": "e lờ", "m": "em mờ", "n": "en nờ",
    "o": "o", "p": "pê", "q": "quy", "r": "e rờ", "s": "ét", "t": "tê", "u": "u", "v": "vê",
    "w": "vê kép", "x": "ích", "y": "i dài", "z": "dét",
}
# Chữ cái ngoài bảng chữ tiếng Việt -> cách viết gần âm nhất
LOANWORD_LETTERS = (("f", "ph"), ("j", "gi"), ("w", "u"), ("z", "d"))

# Dấu thanh (dạng tổ hợp Unicode) -> token thanh điệu; "0" là thanh ngang
TONE_MARKS = {"́": "1", "̀": "2", "̉": "3", "̃": "4", "̣": "5"}
TONES = ["0", "1", "2", "3", "4", "5"]
BASE_LETTERS = list("abcdeghiklmnopqrstuvxy") + ["ă", "â", "đ", "ê", "ô", "ơ", "ư"]
PUNCTUATION = [",", ".", "?", "!"]
# Bộ token cố định của vi_tone_v2; id 0 = blank/pad (chèn giữa các token như add_blank của Matcha)
SYMBOLS_V2 = ["_", " "] + PUNCTUATION + BASE_LETTERS + TONES

_PUNCT_MAP = {";": ",", ":": ",", "…": ".", "\n": "."}
_ITEM_RE = re.compile(r"[a-zăâđêôơừ-ͯ]+|[,.?!]")


def spell_out(text: str) -> str:
    """Chuẩn hóa chính tả trước khi token hóa, giữ nguyên dấu câu.

    Các bước: chuẩn Unicode NFC; thay gạch nối bằng khoảng trắng; đánh vần chữ viết tắt in hoa tối đa
    5 ký tự theo tên chữ cái ("B" -> "bê", "TTS" -> "tê tê ét"); chuyển chữ thường; thay chữ cái ngoại lai
    f/j/w/z bằng âm tiếng Việt gần nhất.

    Args:
        text: Văn bản đã qua normalizer (số, ngày tháng đã viết thành chữ).

    Returns:
        Văn bản chữ thường chỉ gồm chữ cái tiếng Việt, khoảng trắng và dấu câu.
    """
    text = unicodedata.normalize("NFC", text).replace("-", " ")
    text = re.sub(
        r"\b[A-ZĐ]{1,5}\b",
        lambda m: " ".join(LETTER_NAMES[c] for c in m.group(0).lower()),
        text,
    )
    text = text.lower()
    for src, dst in LOANWORD_LETTERS:
        text = text.replace(src, dst)
    return text


def clean_chars(text: str) -> str:
    """Chuẩn hóa văn bản của frontend char_v1: chỉ giữ chữ cái và một khoảng trắng giữa các từ.

    Cũng được dùng để đưa văn bản gốc và kết quả ASR về cùng dạng khi tính CER.

    Args:
        text: Văn bản bất kỳ.

    Returns:
        Chuỗi chữ thường, không dấu câu, không chữ số.
    """
    text = re.sub(r"[\W\d_]+", " ", spell_out(text))
    return re.sub(r"\s+", " ", text).strip()


def syllable_tokens(syllable: str) -> List[str]:
    """Tách một âm tiết thành các chữ cái (đã bỏ dấu thanh) và một token thanh điệu ở cuối.

    Dấu phụ tạo nguyên âm (ă, â, ê, ô, ơ, ư) được giữ trong chữ cái; chỉ năm dấu thanh bị tách ra.
    Ví dụ: "hồ" -> ["h", "ô", "2"]; "gươm" -> ["g", "ư", "ơ", "m", "0"].

    Args:
        syllable: Một âm tiết chữ thường (NFC hoặc NFD).

    Returns:
        Danh sách token; phần tử cuối là thanh điệu "0" (ngang) đến "5" (nặng).
    """
    decomposed = unicodedata.normalize("NFD", syllable)
    tone = next((TONE_MARKS[c] for c in decomposed if c in TONE_MARKS), "0")
    letters = unicodedata.normalize("NFC", "".join(c for c in decomposed if c not in TONE_MARKS))
    return [c for c in letters if c in BASE_LETTERS] + [tone]


def text_to_tokens(text: str, frontend: str = "vi_tone_v2", keep_punctuation: bool = True) -> List[str]:
    """Chuyển văn bản thành danh sách token theo frontend của checkpoint.

    Args:
        text: Văn bản đã qua normalizer.
        frontend: "vi_tone_v2" (tách thanh điệu) hoặc "char_v1" (ký tự mang dấu, checkpoint VIVOS cũ).
        keep_punctuation: Giữ dấu câu (, . ? !) làm token; chỉ bật khi mô hình đã học dấu câu.
            Dấu ; : được quy về dấu phẩy, dấu … và xuống dòng quy về dấu chấm.

    Returns:
        Danh sách token; các âm tiết cách nhau bởi token khoảng trắng " ".
    """
    if frontend == "char_v1":
        return list(clean_chars(text))

    text = spell_out(text)
    for src, dst in _PUNCT_MAP.items():
        text = text.replace(src, dst)
    text = unicodedata.normalize("NFD", text)
    tokens: List[str] = []
    for item in _ITEM_RE.findall(text):
        if item in PUNCTUATION:
            if keep_punctuation and tokens and tokens[-1] not in PUNCTUATION:
                tokens += [" ", item]
            continue
        if tokens:
            tokens.append(" ")
        tokens += syllable_tokens(item)
    while tokens and tokens[0] in PUNCTUATION + [" "]:
        tokens.pop(0)
    return tokens


def tokens_to_ids(tokens: List[str], symbol2id: Dict[str, int]) -> List[int]:
    """Ánh xạ token sang id và chèn blank (id 0) xen giữa, như tùy chọn add_blank=True của Matcha-TTS.

    Token không có trong bộ từ vựng bị bỏ qua.

    Args:
        tokens: Kết quả của `text_to_tokens`.
        symbol2id: Bảng token -> id của checkpoint.

    Returns:
        Danh sách id độ dài 2·n + 1, dạng [0, t1, 0, t2, ..., 0].
    """
    ids = [symbol2id[t] for t in tokens if t in symbol2id]
    result = [0] * (2 * len(ids) + 1)
    result[1::2] = ids
    return result

In [ ]:
import importlib
import json
import math
import random
import time

import numpy as np
import soundfile as sf
from omegaconf import OmegaConf
from matcha.hifigan.config import v1 as HIFIGAN_V1
from matcha.hifigan.denoiser import Denoiser
from matcha.hifigan.env import AttrDict
from matcha.hifigan.models import Generator as HiFiGAN
from matcha.utils.audio import mel_spectrogram

import vi_frontend as vf

importlib.reload(vf)

# Ampere trở lên (A100, L4) dùng bf16, T4 dùng fp16 + GradScaler; CPU (smoke test) không dùng AMP
USE_AMP = HAS_GPU
AMP_DTYPE = torch.bfloat16 if HAS_GPU and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Tham số mel giống checkpoint LJSpeech để tái sử dụng decoder và vocoder HiFi-GAN
SAMPLE_RATE, N_FFT, N_MELS, HOP, WIN, FMIN, FMAX = 22050, 1024, 80, 256, 1024, 0, 8000

PRETRAINED_DIR = f'{WORK_DIR}/pretrained'
MATCHA_LJSPEECH_URL = 'https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/matcha_ljspeech.ckpt'
HIFIGAN_UNIV_URL = 'https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/g_02500000'

EVAL_SENTENCES = [
    'Hôm nay trời trong xanh, gió nhẹ thổi qua những hàng cây ven đường.',
    'Chiến thắng Điện Biên Phủ diễn ra ngày bảy tháng năm năm một nghìn chín trăm năm mươi tư.',
    'Hồ Gươm nằm ở trung tâm thủ đô Hà Nội, gắn liền với truyền thuyết vua Lê Lợi trả gươm báu.',
    'Đáp án đúng là câu bê, vì tác giả muốn nhấn mạnh tình yêu quê hương.',
    'Bạn có biết vì sao lá cây chuyển sang màu vàng vào mùa thu không?',
    'Những người nông dân cần cù đã gặt hái được một vụ mùa bội thu.',
]
if SMOKE_TEST:
    EVAL_SENTENCES = EVAL_SENTENCES[:1]


def download(url, filename):
    path = f'{PRETRAINED_DIR}/{filename}'
    if not os.path.exists(path):
        os.makedirs(PRETRAINED_DIR, exist_ok=True)
        torch.hub.download_url_to_file(url, path)
    return path


def compute_mel(wav, fmax=FMAX):
    """wav (T,) hoặc (B, T) trong [-1, 1] -> log-mel (B, 80, T // 256), cùng công thức với HiFi-GAN."""
    return mel_spectrogram(wav if wav.dim() == 2 else wav[None], N_FFT, N_MELS, SAMPLE_RATE, HOP, WIN, FMIN, fmax)


def text_ids(text):
    """Văn bản -> id token (frontend & bộ token lấy từ meta của dataset, nạp ở mục 5)."""
    tokens = vf.text_to_tokens(text, frontend=meta['frontend'], keep_punctuation=meta['keep_punctuation'])
    return vf.tokens_to_ids(tokens, SYMBOL2ID)


def build_model(hparams):
    return MatchaTTS(**OmegaConf.create(hparams))


def load_matcha(path):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    model = build_model(ckpt['hparams'])
    model.load_state_dict(ckpt['state_dict'])
    return model.to(DEVICE).eval(), ckpt


def load_vocoder(path=None):
    """HiFi-GAN để suy luận (đã bỏ weight norm). Mặc định: bản Universal."""
    path = path or download(HIFIGAN_UNIV_URL, 'hifigan_univ_v1')
    vocoder = HiFiGAN(AttrDict(HIFIGAN_V1)).to(DEVICE)
    vocoder.load_state_dict(torch.load(path, map_location=DEVICE)['generator'])
    vocoder.eval()
    vocoder.remove_weight_norm()
    return vocoder, Denoiser(vocoder, mode='zeros')


def spk_tensor(model, spk_id):
    return torch.tensor([spk_id], dtype=torch.long, device=DEVICE) if model.n_spks > 1 else None


@torch.no_grad()
def synthesize_vi(model, text, vocoder, denoiser=None, spk_id=0, n_timesteps=20, temperature=0.667, length_scale=1.0):
    x = torch.tensor(text_ids(text), dtype=torch.long, device=DEVICE)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=DEVICE)
    was_training = model.training
    model.eval()
    mel = model.synthesise(x, x_lengths, n_timesteps=n_timesteps, temperature=temperature,
                           spks=spk_tensor(model, spk_id), length_scale=length_scale)['mel']
    model.train(was_training)
    audio = vocoder(mel).clamp(-1, 1)
    if denoiser is not None:
        audio = denoiser(audio.squeeze(), strength=0.00025)
    return audio.float().cpu().squeeze().numpy()


def save_atomic(obj, path):
    # Ghi ra file tạm rồi đổi tên: nếu Colab ngắt giữa chừng, file cũ vẫn còn nguyên
    torch.save(obj, path + '.tmp')
    os.replace(path + '.tmp', path)


def show_audio(wav, path=None):
    if path:
        sf.write(path, wav, SAMPLE_RATE)
    try:
        from IPython.display import Audio, display

        display(Audio(wav, rate=SAMPLE_RATE))
    except ImportError:
        pass


print('Vi du token:', '|'.join(vf.text_to_tokens('Hồ Gươm, Hà Nội?')))

## 5. Chuẩn Bị Dữ Liệu

- **Lần đầu:** tải các shard parquet từ Hugging Face, giải mã audio, resample về 22.05 kHz, cắt im lặng hai đầu, **chuẩn hóa âm lượng theo RMS**, tính mel trên GPU, rồi lưu gói dữ liệu lên Drive. Một phần wav được giữ lại để fine-tune vocoder (mục 8).
- **Các lần sau:** chỉ nạp gói từ Drive.
- Dataset nhiều giọng: tự chọn **một** giọng có nhiều dữ liệu nhất. Train một giọng cho kết quả mượt hơn nhiều so với nhiều giọng mà mỗi giọng có ít dữ liệu.

In [ ]:
import io
from multiprocessing import Pool

import librosa
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm

cfg = DATASETS[DATASET]
HF_TOKEN = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata

    HF_TOKEN = HF_TOKEN or userdata.get('HF_TOKEN')
except Exception:
    pass


def load_audio_bytes(raw):
    """bytes audio -> float32 mono 22.05 kHz, đã cắt im lặng và chuẩn hóa âm lượng; None nếu không hợp lệ."""
    try:
        y, sr = sf.read(io.BytesIO(raw), dtype='float32', always_2d=True)
    except Exception:
        return None
    y = y.mean(axis=1)
    if sr != SAMPLE_RATE:
        y = librosa.resample(y, orig_sr=sr, target_sr=SAMPLE_RATE)
    y, _ = librosa.effects.trim(y, top_db=40)
    if not MIN_SECONDS * SAMPLE_RATE <= len(y) <= MAX_SECONDS * SAMPLE_RATE:
        return None
    # Chuẩn hóa theo năng lượng (RMS ~ -20 dBFS) thay vì theo đỉnh để âm lượng các câu đồng đều
    y = y * (0.1 / max(1e-4, float(np.sqrt(np.mean(y ** 2)))))
    peak = float(np.abs(y).max())
    if peak > 0.95:
        y = y * (0.95 / peak)
    return y.astype(np.float32)


def list_shards():
    files = HfApi().list_repo_files(cfg['repo'], repo_type='dataset', token=HF_TOKEN)
    shards = sorted(f for f in files if f.startswith(cfg['shard_prefix']) and f.endswith('.parquet'))
    return shards[:MAX_SHARDS] if MAX_SHARDS else shards


def iter_rows(path, columns):
    for batch in pq.ParquetFile(path).iter_batches(batch_size=64, columns=columns):
        yield from batch.to_pylist()


def pick_speakers(paths):
    """Dataset nhiều giọng: chọn giọng có tổng dung lượng audio lớn nhất (xấp xỉ tổng thời lượng)."""
    if cfg['speaker_col'] is None:
        return None
    if SPEAKERS:
        return set(SPEAKERS)
    sizes = {}
    for path in tqdm(paths, desc='Dem du lieu theo giong'):
        for row in iter_rows(path, [cfg['speaker_col'], cfg['audio_col']]):
            spk = row[cfg['speaker_col']]
            sizes[spk] = sizes.get(spk, 0) + len(row[cfg['audio_col']]['bytes'] or b'')
    best = max(sizes, key=sizes.get)
    print(f'Chon giong {best!r} ({sizes[best] / sum(sizes.values()):.1%} dung luong audio)')
    return {best}


if os.path.exists(DATA_PACK) and os.path.exists(META_PATH) and os.path.exists(VOC_PACK):
    print('Nap du lieu da tien xu ly tu Drive...')
    data = torch.load(DATA_PACK, weights_only=False)
    with open(META_PATH, encoding='utf-8') as f:
        meta = json.load(f)
else:
    shard_paths = [hf_hub_download(cfg['repo'], name, repo_type='dataset', token=HF_TOKEN,
                                   local_dir=f'{WORK_DIR}/hf_data') for name in tqdm(list_shards(), desc='Tai shard')]
    keep_spk = pick_speakers(shard_paths)
    columns = [cfg['text_col'], cfg['audio_col']] + ([cfg['speaker_col']] if cfg['speaker_col'] else [])

    items, voc_items = [], []
    mel_sum, mel_sq_sum, n_frames = 0.0, 0.0, 0
    pool = Pool(N_WORKERS) if N_WORKERS and N_WORKERS > 1 else None
    for path in shard_paths:
        if MAX_UTTS and len(items) >= MAX_UTTS:
            break
        rows = [r for r in iter_rows(path, columns)
                if keep_spk is None or r[cfg['speaker_col']] in keep_spk]
        raws = [r[cfg['audio_col']]['bytes'] for r in rows]
        audios = pool.imap(load_audio_bytes, raws, chunksize=8) if pool else map(load_audio_bytes, raws)
        for r, y in tqdm(zip(rows, audios), total=len(rows), desc=os.path.basename(path)):
            text = r[cfg['text_col']]
            if y is None or len(vf.text_to_tokens(text)) < 2:
                continue
            mel = compute_mel(torch.from_numpy(y).to(DEVICE)).squeeze(0)
            mel_sum += mel.sum().item()
            mel_sq_sum += (mel ** 2).sum().item()
            n_frames += mel.shape[-1]
            spk = r[cfg['speaker_col']] if cfg['speaker_col'] else 'main'
            items.append({'text': text, 'spk': spk, 'mel': mel.half().cpu()})
            if len(voc_items) < VOC_N_UTTS and len(items) % VOC_EVERY == 0:
                voc_items.append({'text': text, 'spk': spk, 'wav': torch.from_numpy((y * 32767).astype(np.int16))})
            if MAX_UTTS and len(items) >= MAX_UTTS:
                break
    if pool:
        pool.close()

    mel_mean = mel_sum / (n_frames * N_MELS)
    meta = {
        'dataset': DATASET,
        'frontend': 'vi_tone_v2',
        # Chỉ học dấu câu khi dữ liệu có dấu câu (InfoRe không có -> engine ngắt cụm theo dấu câu khi suy luận)
        'keep_punctuation': any(c in it['text'] for it in items for c in vf.PUNCTUATION),
        'vocab': vf.SYMBOLS_V2,
        'speakers': sorted(set(it['spk'] for it in items)),
        'mel_mean': mel_mean,
        'mel_std': float(np.sqrt(mel_sq_sum / (n_frames * N_MELS) - mel_mean ** 2)),
    }
    random.Random(1234).shuffle(items)
    data = {'val': items[:N_VAL], 'train': items[N_VAL:]}

    torch.save(data, DATA_PACK)
    torch.save(voc_items, VOC_PACK)
    with open(META_PATH, 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=1)
    print(f'[SAVE] Da luu goi du lieu len Drive: {DATA_PACK}')

SYMBOL2ID = {s: i for i, s in enumerate(meta['vocab'])}
SPK2ID = {s: i for i, s in enumerate(meta['speakers'])}
hours = sum(it['mel'].shape[-1] for it in data['train'] + data['val']) * HOP / SAMPLE_RATE / 3600
print(f"[OK] {len(data['train'])} cau train, {len(data['val'])} cau val ({hours:.1f} gio), "
      f"{len(meta['speakers'])} giong, {len(meta['vocab'])} token, dau cau: {meta['keep_punctuation']}")
print(f"     mel_mean={meta['mel_mean']:.4f}, mel_std={meta['mel_std']:.4f}")
print('     Vi du:', data['train'][0]['text'], '->', '|'.join(vf.text_to_tokens(data['train'][0]['text']))[:120])

## 6. Huấn Luyện Matcha-TTS

- Khởi tạo từ checkpoint LJSpeech: tái sử dụng decoder flow matching và phần lớn text encoder, chỉ học lại embedding token.
- Train theo **số step** (`TOTAL_STEPS`), **không early stopping**. Cứ `SNAPSHOT_EVERY` step lưu một snapshot lên Drive để chọn ở mục 7.
- Cứ `RESUME_EVERY_MIN` phút lưu `matcha_v2_resume.pt` (cuối epoch). Bị ngắt kết nối thì **Run all**, quá trình tự tiếp tục.
- **Ước tính tổng thời gian:** sau epoch đầu, lấy `số phút/epoch × TOTAL_STEPS / số step mỗi epoch`.
- Mốc tham khảo: khoảng 30k step đã nghe rõ chữ; 80k–150k step giọng mượt và ổn định hơn rõ rệt.

In [ ]:
from torch.utils.data import DataLoader, Dataset
from matcha.utils.model import fix_len_compatibility


class MelDataset(Dataset):
    def __init__(self, items):
        self.items = items
        self.lengths = [it['mel'].shape[-1] for it in items]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        x = torch.tensor(text_ids(it['text']), dtype=torch.long)
        y = (it['mel'].float() - meta['mel_mean']) / meta['mel_std']
        return x, y, SPK2ID[it['spk']]


def collate(batch):
    x_lengths = torch.tensor([b[0].shape[0] for b in batch], dtype=torch.long)
    y_lengths = torch.tensor([b[1].shape[-1] for b in batch], dtype=torch.long)
    x = torch.zeros(len(batch), int(x_lengths.max()), dtype=torch.long)
    y = torch.zeros(len(batch), N_MELS, int(fix_len_compatibility(int(y_lengths.max()))))
    for i, (xi, yi, _) in enumerate(batch):
        x[i, :xi.shape[0]] = xi
        y[i, :, :yi.shape[-1]] = yi
    spks = torch.tensor([b[2] for b in batch], dtype=torch.long)
    return x, x_lengths, y, y_lengths, spks


class BucketBatchSampler:
    """Gom các câu có độ dài gần nhau vào cùng batch để giảm padding (nhanh hơn, ít tốn VRAM)."""

    def __init__(self, lengths, batch_size, bucket_batches=50):
        self.lengths, self.batch_size = lengths, batch_size
        self.chunk = batch_size * bucket_batches

    def __iter__(self):
        idx = list(range(len(self.lengths)))
        random.shuffle(idx)
        batches = []
        for s in range(0, len(idx), self.chunk):
            group = sorted(idx[s:s + self.chunk], key=lambda i: self.lengths[i])
            batches += [group[j:j + self.batch_size] for j in range(0, len(group), self.batch_size)]
        random.shuffle(batches)
        return iter(batches)

    def __len__(self):
        n = len(self.lengths)
        return sum(math.ceil(min(self.chunk, n - s) / self.batch_size) for s in range(0, n, self.chunk))


train_set, val_set = MelDataset(data['train']), MelDataset(data['val'])
train_loader = DataLoader(train_set, batch_sampler=BucketBatchSampler(train_set.lengths, BATCH_SIZE),
                          collate_fn=collate, pin_memory=HAS_GPU)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, collate_fn=collate, pin_memory=HAS_GPU)

# ---------------------------- Mô hình: resume hoặc khởi tạo từ LJSpeech ----------------------------
start_epoch, global_step = 1, 0
if os.path.exists(RESUME_CKPT):
    ckpt = torch.load(RESUME_CKPT, map_location='cpu', weights_only=False)
    hparams = ckpt['hparams']
    model = build_model(hparams)
    model.load_state_dict(ckpt['state_dict'])
    start_epoch, global_step = ckpt['epoch'] + 1, ckpt['step']
    print(f'[RESUME] Tiep tuc tu epoch {start_epoch} (step {global_step})')
else:
    ckpt = None
    # PyTorch >= 2.6 mặc định weights_only=True, không đọc được hyper_parameters (omegaconf) của checkpoint Lightning
    lj_ckpt = torch.load(download(MATCHA_LJSPEECH_URL, 'matcha_ljspeech.ckpt'), map_location='cpu', weights_only=False)
    hparams = {k: v for k, v in lj_ckpt['hyper_parameters'].items() if k not in ('optimizer', 'scheduler')}
    hparams = OmegaConf.to_container(OmegaConf.create(hparams), resolve=True)
    hparams.update(n_vocab=len(meta['vocab']), n_spks=len(meta['speakers']), out_size=None,
                   data_statistics={'mel_mean': meta['mel_mean'], 'mel_std': meta['mel_std']})
    model = build_model(hparams)
    # Bộ token tiếng Việt khác LJSpeech -> chỉ nạp các tensor cùng tên & kích thước
    own_state = model.state_dict()
    reused = {k: v for k, v in lj_ckpt['state_dict'].items()
              if k in own_state and own_state[k].shape == v.shape and k not in ('mel_mean', 'mel_std')}
    model.load_state_dict(reused, strict=False)
    print(f'[INIT] Tai su dung {len(reused)}/{len(own_state)} tensor tu checkpoint LJSpeech')
    del lj_ckpt

model = model.to(DEVICE).train()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
if ckpt is not None:
    optimizer.load_state_dict(ckpt['optimizer'])
    scaler.load_state_dict(ckpt['scaler'])
del ckpt

vocoder, denoiser = load_vocoder()
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M tham so | batch {BATCH_SIZE} | '
      f'{len(train_loader)} step/epoch | muc tieu {TOTAL_STEPS} step')


def run_batch(batch):
    x, x_lengths, y, y_lengths, spks = (t.to(DEVICE, non_blocking=True) for t in batch)
    with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
        dur_loss, prior_loss, diff_loss, _ = model(x, x_lengths, y, y_lengths,
                                                   spks=spks if model.n_spks > 1 else None)
    return dur_loss.float(), prior_loss.float(), diff_loss.float()


@torch.no_grad()
def evaluate():
    model.eval()
    totals = np.zeros(3)
    for batch in val_loader:
        totals += [l.item() for l in run_batch(batch)]
    model.train()
    return totals / len(val_loader)


def snapshot(epoch):
    # Snapshot chứa cả meta (bộ token, frontend) -> engine suy luận chỉ cần đúng một file .pt
    return {'hparams': hparams, 'state_dict': model.state_dict(), 'meta': meta, 'epoch': epoch, 'step': global_step}


# ---------------------------- Vòng lặp huấn luyện ----------------------------
epoch, last_resume_save = start_epoch, time.time()
while global_step < TOTAL_STEPS:
    t0 = time.time()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch} (step {global_step}/{TOTAL_STEPS})')
    for batch in pbar:
        dur_loss, prior_loss, diff_loss = run_batch(batch)
        loss = dur_loss + prior_loss + diff_loss
        optimizer.zero_grad(set_to_none=True)
        if not torch.isfinite(loss):
            print(f'[WARN] Loss khong hop le o step {global_step}, bo qua batch')
            continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        global_step += 1
        pbar.set_postfix(loss=f'{loss.item():.3f}', dur=f'{dur_loss.item():.3f}',
                         prior=f'{prior_loss.item():.3f}', diff=f'{diff_loss.item():.3f}')

        if global_step % SNAPSHOT_EVERY == 0:
            save_atomic(snapshot(epoch), f'{SNAPSHOT_DIR}/step_{global_step:06d}.pt')
        if global_step % SAMPLE_EVERY == 0:
            wav = synthesize_vi(model, EVAL_SENTENCES[0], vocoder, denoiser)
            show_audio(wav, f'{SAMPLES_DIR}/step_{global_step:06d}.wav')
        if global_step >= TOTAL_STEPS:
            break

    val_dur, val_prior, val_diff = evaluate()
    print(f'==> Epoch {epoch} | step {global_step} | {(time.time() - t0) / 60:.1f} phut | '
          f'val loss {val_dur + val_prior + val_diff:.4f} (dur {val_dur:.3f}, prior {val_prior:.3f}, diff {val_diff:.3f})')
    if time.time() - last_resume_save >= RESUME_EVERY_MIN * 60 or global_step >= TOTAL_STEPS:
        save_atomic({**snapshot(epoch), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict()},
                    RESUME_CKPT)
        last_resume_save = time.time()
    epoch += 1

print(f'\n[THANH CONG] Hoan tat {global_step} step!')

## 7. Chọn Checkpoint Bằng CER (PhoWhisper) & Nghe Thử

Val loss của flow matching không phản ánh chất giọng, nên mục này chấm các snapshot mới nhất bằng **CER**: ASR tiếng Việt PhoWhisper nghe lại giọng tổng hợp, CER thấp nghĩa là đọc rõ và đúng thanh. Snapshot tốt nhất được copy thành `matcha_vi_v2.pt`.

CER chỉ đo **độ rõ**. Hãy nghe các câu mẫu: nếu thấy snapshot khác tự nhiên hơn, đặt `CHOSEN_SNAPSHOT` rồi chạy lại cell. Đã có `matcha_vi_v2.pt` thì cell giữ nguyên lựa chọn cũ, trừ khi đặt `REEVALUATE = True`.

In [ ]:
import glob
import shutil

from scipy.signal import resample_poly
from transformers import pipeline

CHOSEN_SNAPSHOT = None   # ví dụ f'{SNAPSHOT_DIR}/step_120000.pt' để chọn thủ công
REEVALUATE = False       # True để chấm lại khi đã có matcha_vi_v2.pt (vd. sau khi tăng TOTAL_STEPS và train thêm)

asr = pipeline('automatic-speech-recognition', model='vinai/PhoWhisper-base', device=0 if HAS_GPU else -1,
               generate_kwargs={'language': 'vi', 'task': 'transcribe'})


def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def cer(ref, hyp):
    ref, hyp = vf.clean_chars(ref), vf.clean_chars(hyp)
    return edit_distance(ref, hyp) / max(1, len(ref))


eval_vocoder, eval_denoiser = load_vocoder()
if os.path.exists(FINAL_CKPT) and not (REEVALUATE or CHOSEN_SNAPSHOT):
    # Giữ lựa chọn cũ khi Run all lại, để mục 8 (vocoder) không phải fine-tune lại từ đầu
    print(f'[GIU NGUYEN] Da co {FINAL_CKPT}. Dat REEVALUATE = True de cham lai (vd. sau khi train them).')
elif CHOSEN_SNAPSHOT:
    shutil.copy(CHOSEN_SNAPSHOT, FINAL_CKPT)
    print(f'[CHON THU CONG] {CHOSEN_SNAPSHOT} -> {FINAL_CKPT}')
else:
    torch.manual_seed(0)
    results = []
    for path in sorted(glob.glob(f'{SNAPSHOT_DIR}/step_*.pt'))[-N_EVAL_SNAPSHOTS:]:
        tts, ck = load_matcha(path)
        errs = []
        for sentence in EVAL_SENTENCES:
            wav = synthesize_vi(tts, sentence, eval_vocoder, eval_denoiser)
            hyp = asr(resample_poly(wav, 16000, SAMPLE_RATE).astype(np.float32))['text']
            errs.append(cer(sentence, hyp))
        results.append((float(np.mean(errs)), ck['step'], path))
        print(f"step {ck['step']:>7}: CER {results[-1][0]:.3f}")
        del tts

    # CER thấp nhất; bằng nhau thì ưu tiên snapshot train lâu hơn
    best_cer, best_step, best_path = min(results, key=lambda r: (round(r[0], 3), -r[1]))
    shutil.copy(best_path, FINAL_CKPT)
    print(f'\n[CHON] {os.path.basename(best_path)} -> {FINAL_CKPT} (CER {best_cer:.3f})')

tts, _ = load_matcha(FINAL_CKPT)
for i, sentence in enumerate(EVAL_SENTENCES):
    print(sentence)
    show_audio(synthesize_vi(tts, sentence, eval_vocoder, eval_denoiser), f'{SAMPLES_DIR}/final_univ_{i}.wav')

## 8. Fine-Tune Vocoder HiFi-GAN

HiFi-GAN Universal học trên mel của audio thật, nhưng mel do Matcha sinh ra hơi "mờ" hơn, nên giọng bị rè, kim loại. Cách khắc phục chuẩn (như Tacotron 2, Glow-TTS):

1. **Teacher forcing:** dùng `matcha_vi_v2.pt` sinh mel **khớp từng khung** với câu thật. Căn chỉnh văn bản và âm thanh bằng MAS, rồi chạy decoder flow matching.
2. **Fine-tune HiFi-GAN** để biến mel do mô hình sinh ra thành đúng waveform thật. Generator khởi tạo từ bản Universal. Discriminator học từ đầu, nên `VOC_DISC_WARMUP` step đầu generator chỉ dùng mel loss.

Nếu chọn lại checkpoint ở mục 7, mel teacher-forced sẽ tự sinh lại.

In [ ]:
import torch.nn.functional as F
import matcha.utils.monotonic_align as monotonic_align
from matcha.utils.model import denormalize, sequence_mask

voc_items = torch.load(VOC_PACK, weights_only=False)
tts, tts_ckpt = load_matcha(FINAL_CKPT)


@torch.no_grad()
def teacher_forced_mel(model, text, spk, wav):
    """Mel do decoder Matcha sinh ra nhưng theo đúng căn chỉnh (MAS) của câu thật -> khớp từng khung với wav."""
    y = (compute_mel(wav.to(DEVICE)) - model.mel_mean) / model.mel_std          # (1, 80, T)
    n_frames = y.shape[-1]
    x = torch.tensor(text_ids(text), dtype=torch.long, device=DEVICE)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=DEVICE)
    spks = model.spk_emb(spk_tensor(model, SPK2ID[spk])) if model.n_spks > 1 else None
    mu_x, _, x_mask = model.encoder(x, x_lengths, spks)

    padded = fix_len_compatibility(n_frames)
    y = F.pad(y, (0, padded - n_frames))
    y_mask = sequence_mask(torch.tensor([n_frames], device=DEVICE), padded).unsqueeze(1).to(x_mask.dtype)
    attn_mask = x_mask.unsqueeze(-1) * y_mask.unsqueeze(2)
    # Monotonic Alignment Search, giống hệt MatchaTTS.forward
    factor = -0.5 * torch.ones_like(mu_x)
    log_prior = (torch.matmul(factor.transpose(1, 2), y ** 2)
                 - torch.matmul(2.0 * (factor * mu_x).transpose(1, 2), y)
                 + torch.sum(factor * mu_x ** 2, 1).unsqueeze(-1)
                 - 0.5 * math.log(2 * math.pi) * model.n_feats)
    attn = monotonic_align.maximum_path(log_prior, attn_mask.squeeze(1))
    mu_y = torch.matmul(attn.transpose(1, 2), mu_x.transpose(1, 2)).transpose(1, 2)
    mel = model.decoder(mu_y, y_mask, VOC_TF_TIMESTEPS, 0.667, spks)
    return denormalize(mel, model.mel_mean, model.mel_std)[0, :, :n_frames]


tf_pack = torch.load(VOC_TF_PACK, weights_only=False) if os.path.exists(VOC_TF_PACK) else None
if tf_pack is None or tf_pack['source_step'] != tts_ckpt['step']:
    pairs = []
    for it in tqdm(voc_items, desc='Sinh mel teacher-forced'):
        wav = it['wav'].float() / 32768.0
        mel = teacher_forced_mel(tts, it['text'], it['spk'], wav)
        pairs.append({'mel': mel.half().cpu(), 'wav': it['wav'][:mel.shape[-1] * HOP]})
    tf_pack = {'source_step': tts_ckpt['step'], 'pairs': pairs}
    save_atomic(tf_pack, VOC_TF_PACK)
del tts
print(f"[OK] {len(tf_pack['pairs'])} cap (mel sinh ra, wav that) tu checkpoint step {tf_pack['source_step']}")

In [ ]:
import itertools

from matcha.hifigan.models import (MultiPeriodDiscriminator, MultiScaleDiscriminator, discriminator_loss,
                                   feature_loss, generator_loss)

SEG = HIFIGAN_V1['segment_size']      # 8192 mẫu = 32 khung mel
SEG_FRAMES = SEG // HOP
LOG_SILENCE = math.log(1e-5)          # giá trị log-mel của khoảng lặng (dùng để pad)


class VocDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        mel, wav = self.pairs[i]['mel'].float(), self.pairs[i]['wav'].float() / 32768.0
        if mel.shape[-1] >= SEG_FRAMES:
            start = random.randint(0, mel.shape[-1] - SEG_FRAMES)
            mel, wav = mel[:, start:start + SEG_FRAMES], wav[start * HOP:(start + SEG_FRAMES) * HOP]
        mel = F.pad(mel, (0, SEG_FRAMES - mel.shape[-1]), value=LOG_SILENCE)
        wav = F.pad(wav, (0, SEG - wav.shape[-1]))
        return mel, wav


voc_loader = DataLoader(VocDataset(tf_pack['pairs']), batch_size=VOC_BATCH, shuffle=True, drop_last=True,
                        num_workers=2 if HAS_GPU else 0, pin_memory=HAS_GPU)

generator = HiFiGAN(AttrDict(HIFIGAN_V1)).to(DEVICE)
mpd, msd = MultiPeriodDiscriminator().to(DEVICE), MultiScaleDiscriminator().to(DEVICE)
optim_g = torch.optim.AdamW(generator.parameters(), VOC_LR, betas=(0.8, 0.99))
optim_d = torch.optim.AdamW(itertools.chain(msd.parameters(), mpd.parameters()), VOC_LR, betas=(0.8, 0.99))
voc_step = 0
if os.path.exists(VOC_RESUME):
    state = torch.load(VOC_RESUME, map_location=DEVICE, weights_only=False)
    if state['source_step'] == tf_pack['source_step']:
        for module, key in ((generator, 'generator'), (mpd, 'mpd'), (msd, 'msd'), (optim_g, 'optim_g'), (optim_d, 'optim_d')):
            module.load_state_dict(state[key])
        voc_step = state['step']
        print(f'[RESUME] Vocoder tiep tuc tu step {voc_step}')
    del state
if voc_step == 0:
    generator.load_state_dict(torch.load(download(HIFIGAN_UNIV_URL, 'hifigan_univ_v1'), map_location=DEVICE)['generator'])
    print('[INIT] Generator khoi tao tu HiFi-GAN Universal, discriminator hoc tu dau')
generator.train(), mpd.train(), msd.train()


def save_vocoder():
    save_atomic({'generator': generator.state_dict(), 'mpd': mpd.state_dict(), 'msd': msd.state_dict(),
                 'optim_g': optim_g.state_dict(), 'optim_d': optim_d.state_dict(), 'step': voc_step,
                 'source_step': tf_pack['source_step']}, VOC_RESUME)
    # Cùng định dạng checkpoint HiFi-GAN gốc -> engine nạp được như bản Universal
    save_atomic({'generator': generator.state_dict(), 'step': voc_step, 'source_step': tf_pack['source_step']}, FINAL_VOC)


tts, _ = load_matcha(FINAL_CKPT)
last_voc_save = time.time()
pbar = tqdm(total=VOC_STEPS, initial=voc_step, desc='Fine-tune HiFi-GAN')
while voc_step < VOC_STEPS:
    for mel, wav in voc_loader:
        mel, y = mel.to(DEVICE, non_blocking=True), wav.to(DEVICE, non_blocking=True).unsqueeze(1)
        y_hat = generator(mel)
        y_mel = compute_mel(y.squeeze(1), fmax=None)          # mel loss trên toàn dải tần như HiFi-GAN gốc
        y_hat_mel = compute_mel(y_hat.squeeze(1), fmax=None)

        # Discriminator
        optim_d.zero_grad(set_to_none=True)
        y_df_r, y_df_g, _, _ = mpd(y, y_hat.detach())
        y_ds_r, y_ds_g, _, _ = msd(y, y_hat.detach())
        loss_d = discriminator_loss(y_df_r, y_df_g)[0] + discriminator_loss(y_ds_r, y_ds_g)[0]
        loss_d.backward()
        optim_d.step()

        # Generator
        optim_g.zero_grad(set_to_none=True)
        loss_mel = F.l1_loss(y_mel, y_hat_mel) * 45
        loss_g = loss_mel
        if voc_step >= VOC_DISC_WARMUP:
            _, y_df_g, fmap_f_r, fmap_f_g = mpd(y, y_hat)
            _, y_ds_g, fmap_s_r, fmap_s_g = msd(y, y_hat)
            loss_g = (loss_mel + feature_loss(fmap_f_r, fmap_f_g) + feature_loss(fmap_s_r, fmap_s_g)
                      + generator_loss(y_df_g)[0] + generator_loss(y_ds_g)[0])
        loss_g.backward()
        optim_g.step()

        voc_step += 1
        pbar.update(1)
        pbar.set_postfix(mel=f'{loss_mel.item() / 45:.3f}', d=f'{loss_d.item():.3f}', g=f'{loss_g.item():.3f}')
        if time.time() - last_voc_save >= VOC_RESUME_EVERY_MIN * 60:
            save_vocoder()
            last_voc_save = time.time()
        if voc_step % VOC_SAMPLE_EVERY == 0:
            generator.eval()
            show_audio(synthesize_vi(tts, EVAL_SENTENCES[0], generator), f'{SAMPLES_DIR}/vocoder_step_{voc_step:06d}.wav')
            generator.train()
        if voc_step >= VOC_STEPS:
            break
save_vocoder()
pbar.close()
print(f'[THANH CONG] Vocoder fine-tune xong {voc_step} step -> {FINAL_VOC}')

## 9. Nghe Thử Kết Quả Cuối & Xuất Mô Hình

So sánh cùng câu với **HiFi-GAN Universal** và **HiFi-GAN fine-tune**. Sau đó tải hai file trong `MyDrive/Colab Notebooks/TTS/export/` về thư mục `checkpoints/` của dự án:

- `matcha_vi_v2.pt`: mô hình Matcha, đã kèm bộ token và frontend.
- `matcha_vi_v2_hifigan.pt`: vocoder fine-tune.

`src/matcha_engine.py` tự ưu tiên `matcha_vi_v2.pt` và tự dùng vocoder `<tên checkpoint>_hifigan.pt` đi kèm. Kiểm tra lại bằng `python scripts/audition_speakers.py --checkpoint checkpoints/matcha_vi_v2.pt` và `python scripts/benchmark.py`.

In [ ]:
tts, ck = load_matcha(FINAL_CKPT)
voc_univ, den_univ = load_vocoder()
voc_ft, den_ft = load_vocoder(FINAL_VOC)
print(f"Matcha step {ck['step']} | frontend {ck['meta']['frontend']} | {len(ck['meta']['speakers'])} giong")
for i, sentence in enumerate(EVAL_SENTENCES):
    print(f'\n{sentence}')
    print('  HiFi-GAN Universal:')
    show_audio(synthesize_vi(tts, sentence, voc_univ, den_univ), f'{SAMPLES_DIR}/compare_{i}_univ.wav')
    print('  HiFi-GAN fine-tune:')
    show_audio(synthesize_vi(tts, sentence, voc_ft, den_ft), f'{SAMPLES_DIR}/compare_{i}_finetune.wav')

print(f'\nTai 2 file trong {EXPORT_DIR} ve thu muc checkpoints/ cua du an:\n  {FINAL_CKPT}\n  {FINAL_VOC}')